In [1]:
import pandas as pd
import re
from collections import Counter
from pathlib import Path

# Load the original commit records and project statistics
commits = pd.read_csv(
    f"sydlwils_project_summary.csv",
    sep=";"
)

stats = pd.read_csv(
    f"sydlwils_project_stats.csv"
)

print("Commit columns:", commits.columns.tolist())
print("Stats columns:", stats.columns.tolist())
print("Projects in commits:", commits["project"].nunique())

Commit columns: ['project', 'commit', 'author', 'time', 'message']
Stats columns: ['project', 'LongestGapStart', 'LongestGapEnd', 'LongestGapLength', 'NumCommitsAfterLastGap', 'NumberOfGapsInTimeline', 'ActivityPattern']
Projects in commits: 10


In [2]:
# Convert commit timestamps to UTC dates
if pd.api.types.is_numeric_dtype(commits["time"]):
    commits["commit_date"] = pd.to_datetime(
        commits["time"], unit="s", utc=True
    )
else:
    commits["commit_date"] = pd.to_datetime(
        commits["time"], utc=True
    )

commits["commit_month"] = commits["commit_date"].dt.strftime("%Y-%m")

# Remove accidental whitespace from project identifiers
commits["project"] = commits["project"].astype(str).str.strip()
stats["project"] = stats["project"].astype(str).str.strip()

# Verify that the project names match between files
print("Projects in stats but not commits:")
print(sorted(set(stats["project"]) - set(commits["project"])))

print("Projects in commits but not stats:")
print(sorted(set(commits["project"]) - set(stats["project"])))

Projects in stats but not commits:
[]
Projects in commits but not stats:
[]


In [4]:
gap_commits = []

for _, project_row in stats.iterrows():
    project = project_row["project"]

    project_commits = commits[
        commits["project"] == project
    ].copy()

    # Skip projects whose gap dates are unavailable
    gap_start = project_row.get("LongestGapStart")
    gap_end = project_row.get("LongestGapEnd")

    if pd.isna(gap_start) or pd.isna(gap_end):
        print(f"Missing gap dates for {project}; skipping.")
        continue

    gap_start = str(gap_start)[:7]
    gap_end = str(gap_end)[:7]

    # Last 10 commits before the first zero-commit month
    before = project_commits[
        project_commits["commit_month"] < gap_start
    ].sort_values("commit_date", ascending=False).head(10)

    # First 10 commits after the final zero-commit month
    after = project_commits[
        project_commits["commit_month"] > gap_end
    ].sort_values("commit_date", ascending=True).head(10)

    # Label each group
    before = before.copy()
    before.insert(0, "pre/post", "pre")

    after = after.copy()
    after.insert(0, "pre/post", "post")

    gap_commits.extend([before, after])

# Combine all projects
gap_commits_df = pd.concat(
    gap_commits,
    ignore_index=True
) if gap_commits else pd.DataFrame()

# Keep the requested columns, in the requested order
gap_commits_df = gap_commits_df[
    ["pre/post", "project", "commit", "author", "time", "message"]
]

# Save the Part 3 commit dataset
gap_commits_df.to_csv(
    f"sydlwils_project_gap_commits.csv",
    sep=";",
    index=False
)

print("Saved:", f"sydlwils_project_gap_commits.csv")
print(gap_commits_df.groupby(["project", "pre/post"]).size())

Saved: sydlwils_project_gap_commits.csv
project                                       pre/post
google_uis-rnn                                post        10
                                              pre         10
inab_wes-backend                              post        10
                                              pre         10
knossos-project_knossos                       post        10
                                              pre         10
leftygray_cascade_calculations                post        10
                                              pre         10
nrel_openstudio-server                        post        10
                                              pre         10
openmd_openmd                                 post        10
                                              pre         10
parflow_parflow                               post        10
                                              pre         10
project-konda_pattern-based-quality-analysis  post 

In [6]:
def classify_theme(row):
    message = str(row["message"]).lower()
    author = str(row["author"]).lower()

    # Automated contributions
    if (
        any(x in author for x in [
            "dependabot", "renovate", "github-actions",
            "github action", "bot"
        ])
        or any(x in message for x in [
            "dependabot", "renovate", "automated",
            "auto-format", "pre-commit.ci"
        ])
    ):
        return "Automated bot contributions"

    # Security
    if any(x in message for x in [
        "security", "vulnerability", "vulnerabilities",
        "sanitize", "exploit"
    ]):
        return "Security patches"

    # Dependency changes
    if any(x in message for x in [
        "dependency", "dependencies",
        "bump ", "upgrade package", "requirements.txt",
        "pyproject.toml", "update package"
    ]):
        return "Dependency updates"

    # Releases
    if any(x in message for x in [
        "release", "version bump", "prepare version",
        "changelog", "publish", "update", "updates"
    ]):
        return "Release"

    # Documentation
    if any(x in message for x in [
        "docs", "documentation", "readme",
        "typo", "docstring"
    ]):
        return "Documentation updates"

    # Bug fixes
    if any(x in message for x in [
        "fix", "bug", "correct", "resolve",
        "patch", "repair"
    ]):
        return "Bug fixes"

    # Branching
    if any(x in message for x in [
        "merge", "merging", "pull request",
        "merge pull request", "merge request"
    ]):
        return "Branch Handling"

    # Features
    if any(x in message for x in [
        "feature", "introduce",
        "support ", "new ", "enable", "improve"
    ]):
        return "Feature development"

    return "Other"


gap_commits_df["Theme"] = gap_commits_df.apply(
    classify_theme, axis=1
)

gap_commits_df[
    ["project", "pre/post", "author", "message", "Theme"]
].head(20)

,project,pre/post,author,message,Theme
0,nrel_openstudio-server,pre,TJC <tjc.coleman@gmail.com>,Merge pull request #672 from NREL/UO_barebones...,Branch Handling
1,nrel_openstudio-server,pre,TJC <tjc.coleman@gmail.com>,Merge pull request #672 from NREL/UO_barebones...,Branch Handling
2,nrel_openstudio-server,pre,brianlball <brian.ball@nrel.gov>,add URBANopt template OSA for single_run\n,Other
3,nrel_openstudio-server,pre,brianlball <brian.ball@nrel.gov>,add URBANopt template OSA for single_run\n,Other
4,nrel_openstudio-server,pre,TJC <tjc.coleman@gmail.com>,Merge pull request #671 from NREL/UO_update\n\...,Release
5,nrel_openstudio-server,pre,TJC <tjc.coleman@gmail.com>,Merge pull request #671 from NREL/UO_update\n\...,Release
6,nrel_openstudio-server,pre,TJC <tjc.coleman@gmail.com>,Merge pull request #668 from NREL/sobol\n\nSob...,Bug fixes
7,nrel_openstudio-server,pre,TJC <tjc.coleman@gmail.com>,Merge pull request #668 from NREL/sobol\n\nSob...,Bug fixes
8,nrel_openstudio-server,pre,brianlball <brian.ball@nrel.gov>,UO 0.8.0\n,Other
9,nrel_openstudio-server,pre,brianlball <brian.ball@nrel.gov>,UO 0.8.0\n,Other


In [7]:
def top_two_themes(group):
    counts = Counter(group["Theme"].dropna())

    # Most frequent themes first; alphabetical order breaks ties
    ranked = sorted(
        counts.items(),
        key=lambda item: (-item[1], item[0])
    )

    return ":".join(theme for theme, count in ranked[:2])


theme_results = []

for project in stats["project"]:
    project_data = gap_commits_df[
        gap_commits_df["project"] == project
    ]

    before = project_data[
        project_data["pre/post"] == "pre"
    ]

    after = project_data[
        project_data["pre/post"] == "post"
    ]

    theme_results.append({
        "project": project,
        "BeforeThemes": top_two_themes(before),
        "AfterThemes": top_two_themes(after)
    })

themes_df = pd.DataFrame(theme_results)

themes_df

,project,BeforeThemes,AfterThemes
0,nrel_openstudio-server,Other:Branch Handling,Other:Bug fixes
1,knossos-project_knossos,Other:Bug fixes,Other
2,parflow_parflow,Bug fixes:Other,Bug fixes:Other
3,openmd_openmd,Other:Bug fixes,Other
4,project-konda_pattern-based-quality-analysis,Bug fixes:Feature development,Bug fixes:Other
5,inab_wes-backend,Bug fixes:Branch Handling,Bug fixes:Dependency updates
6,wistaria_materiappsinstaller,Branch Handling:Other,Branch Handling:Other
7,leftygray_cascade_calculations,Release,Release
8,wtmatlock_flanker,Other:Release,Release
9,google_uis-rnn,Other:Bug fixes,Bug fixes:Dependency updates


In [8]:
# Add the new theme classifications
stats = stats.merge(
    themes_df,
    on="project",
    how="left",
    validate="one_to_one"
)

# Save the updated statistics file
stats.to_csv(
    f"sydlwils_project_stats.csv",
    index=False
)

print("Updated project statistics:")
display(
    stats[["project", "BeforeThemes", "AfterThemes"]]
)

Updated project statistics:


,project,BeforeThemes,AfterThemes
0,nrel_openstudio-server,Other:Branch Handling,Other:Bug fixes
1,knossos-project_knossos,Other:Bug fixes,Other
2,parflow_parflow,Bug fixes:Other,Bug fixes:Other
3,openmd_openmd,Other:Bug fixes,Other
4,project-konda_pattern-based-quality-analysis,Bug fixes:Feature development,Bug fixes:Other
5,inab_wes-backend,Bug fixes:Branch Handling,Bug fixes:Dependency updates
6,wistaria_materiappsinstaller,Branch Handling:Other,Branch Handling:Other
7,leftygray_cascade_calculations,Release,Release
8,wtmatlock_flanker,Other:Release,Release
9,google_uis-rnn,Other:Bug fixes,Bug fixes:Dependency updates


In [10]:
import pandas as pd

# Load the existing CSV
df = pd.read_csv("sydlwils_project_stats.csv")

# Project ID -> hypothesized reason for inactivity
gap_reasons = {
    "parflow_parflow": (
        "The approximately two-month gap may have been used to prepare "
        "for a new version release, as the first commit afterward was a release."
    ),
    "inab_wes-backend": (
        "Most commits after the gap were minor or cosmetic fixes, and activity "
        "was dominated by one contributor. The break may reflect competing "
        "priorities or a temporary pause, but the cause is unconfirmed."
    ),
    "knossos-project_knossos": (
        "The repository had several gaps of one to two months without a clear "
        "pattern. Surrounding commits primarily involved debugging and bug fixes, "
        "suggesting intermittent maintenance."
    ),
    "project-konda_pattern-based-quality-analysis": (
        "Development appears to occur in short bursts resembling sprints. "
        "The approximately one-month gap may reflect the project's normal "
        "development cadence rather than abandonment."
    ),
    "wistaria_materiappsinstaller": (
        "Branch merges surrounded the gap, suggesting that work may have been "
        "distributed across branches. The first commits afterward focused on "
        "library updates."
    ),
    "wtmatlock_flanker": (
        "The final commits before the gap primarily updated the README, while "
        "commits afterward focused on testing files. This may reflect a transition "
        "toward testing or final validation, although the cause of the gap is unclear."
    ),
    "openmd_openmd": (
        "The first commits after the gap addressed compatibility with a NumPy "
        "upgrade. Dependency maintenance may have motivated renewed activity, "
        "but the cause of the preceding gap is unconfirmed."
    ),
    "google_uis-rnn": (
        "The repository is archived and read-only, and the gap lasted approximately "
        "two years. Subsequent commits primarily updated package requirements, "
        "possibly to address dependency compatibility."
    ),
    "leftygray_cascade_calculations": (
        "Commits after the gap primarily added publications and data and updated "
        "the README, suggesting renewed attention to research documentation "
        "and project resources."
    ),
    "nrel_openstudio-server": (
        "Commits before the gap primarily involved branch merges, while commits "
        "afterward focused on package and requirements upgrades. The interruption "
        "may reflect a change in development or integration activity."
    ),
}

# Identify the project ID column
id_column = "project"

if id_column not in df.columns:
    raise ValueError(
        f"Column '{id_column}' not found. Available columns: {list(df.columns)}"
    )

# Ensure the target column exists
if "HypothesizedGapReason" not in df.columns:
    df["HypothesizedGapReason"] = pd.NA

# Update matching projects only
for project_id, reason in gap_reasons.items():
    mask = df[id_column].astype(str).str.strip().eq(project_id)
    if mask.any():
        df.loc[mask, "HypothesizedGapReason"] = reason
    else:
        print(f"Project not found in CSV: {project_id}")

# Save the updated CSV
df.to_csv("sydlwils_project_stats.csv", index=False)

print("Updated sydlwils_project_stats.csv")

Updated sydlwils_project_stats.csv


In [11]:
print("Stats Columns:", df.columns.tolist())
print("Unique projects:", df['project'].tolist())

recovery_analysis = []

for project in df['project']:
    # Get commits before and after for this project
    p_commits_pre = gap_commits_df[(gap_commits_df['project'] == project) & (gap_commits_df['pre/post'] == 'pre')]
    p_commits_post = gap_commits_df[(gap_commits_df['project'] == project) & (gap_commits_df['pre/post'] == 'post')]

    # Unique authors
    authors_pre = set(p_commits_pre['author'].dropna().unique())
    authors_post = set(p_commits_post['author'].dropna().unique())

    # Overlap
    common_authors = authors_pre.intersection(authors_post)
    new_authors = authors_post - authors_pre

    print(f"Project: {project}")
    print(f"  Pre-gap authors: {authors_pre}")
    print(f"  Post-gap authors: {authors_post}")
    print(f"  Common authors: {common_authors}")
    print(f"  New authors: {new_authors}")
    print("-" * 40)

Stats Columns: ['project', 'LongestGapStart', 'LongestGapEnd', 'LongestGapLength', 'NumCommitsAfterLastGap', 'NumberOfGapsInTimeline', 'ActivityPattern', 'BeforeThemes', 'AfterThemes', 'HypothesizedGapReason']
Unique projects: ['nrel_openstudio-server', 'knossos-project_knossos', 'parflow_parflow', 'openmd_openmd', 'project-konda_pattern-based-quality-analysis', 'inab_wes-backend', 'wistaria_materiappsinstaller', 'leftygray_cascade_calculations', 'wtmatlock_flanker', 'google_uis-rnn']
Project: nrel_openstudio-server
  Pre-gap authors: {'brianlball <brian.ball@nrel.gov>', 'TJC <tjc.coleman@gmail.com>'}
  Post-gap authors: {'Tim Coleman <tjc.coleman@gmail.com>'}
  Common authors: set()
  New authors: {'Tim Coleman <tjc.coleman@gmail.com>'}
----------------------------------------
Project: knossos-project_knossos
  Pre-gap authors: {'Norbert Pfeiler <norbert.pfeiler+git@gmail.com>'}
  Post-gap authors: {'Norbert Pfeiler <norbert.pfeiler+git@gmail.com>'}
  Common authors: {'Norbert Pfeiler

In [12]:
recovery_data = {
    "parflow_parflow": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Recovered to release a new version; first post-gap commit is a version release.",
        "WhoRecovered": "Same contributors (the core development team continued commits)."
    },
    "inab_wes-backend": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Recovered for minor, cosmetic, and dependency fixes.",
        "WhoRecovered": "Same contributor (activity pre- and post-gap is dominated by the same single author)."
    },
    "knossos-project_knossos": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Recovered to apply bug fixes and maintain debugging support.",
        "WhoRecovered": "Same contributors (intermittent maintenance by the existing development team)."
    },
    "project-konda_pattern-based-quality-analysis": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Recovered following a normal cyclical sprint gap to resume development.",
        "WhoRecovered": "Same contributors (the same core developers returned in a new sprint)."
    },
    "wistaria_materiappsinstaller": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Recovered to update libraries and merge feature branches.",
        "WhoRecovered": "Same contributors (collaborative branch merging by original authors)."
    },
    "wtmatlock_flanker": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Recovered to update test suites and perform final validation before project completion.",
        "WhoRecovered": "Same contributors (the same authors returned to finalize testing)."
    },
    "openmd_openmd": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Recovered to update code compatibility for a NumPy dependency upgrade.",
        "WhoRecovered": "Same contributors (existing maintainers resolved compatibility issues)."
    },
    "google_uis-rnn": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Recovered briefly to apply package/dependency updates to keep the archived code runnable.",
        "WhoRecovered": "New and existing contributors (some automated/community package maintenance updates)."
    },
    "leftygray_cascade_calculations": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Recovered to add research publications, data, and update documentation.",
        "WhoRecovered": "Same contributors (the main researcher returned to update project documentation)."
    },
    "nrel_openstudio-server": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Recovered to perform SDK integration, changelog maintenance, and package upgrades.",
        "WhoRecovered": "Same contributors (Tim Coleman / TJC continued primary integration work)."
    }
}

# Initialize the new columns
df["HasRecovered"] = pd.NA
df["WhyRecovered"] = pd.NA
df["WhoRecovered"] = pd.NA

for proj, details in recovery_data.items():
    mask = df["project"].astype(str).str.strip().eq(proj)
    if mask.any():
        df.loc[mask, "HasRecovered"] = details["HasRecovered"]
        df.loc[mask, "WhyRecovered"] = details["WhyRecovered"]
        df.loc[mask, "WhoRecovered"] = details["WhoRecovered"]

# Save the final table
df.to_csv("sydlwils_project_stats.csv", index=False)
print("Saved final updated stats table to sydlwils_project_stats.csv")
display(df[["project", "HypothesizedGapReason", "HasRecovered", "WhyRecovered", "WhoRecovered"]])

Saved final updated stats table to sydlwils_project_stats.csv


,project,HypothesizedGapReason,HasRecovered,WhyRecovered,WhoRecovered
0,nrel_openstudio-server,Commits before the gap primarily involved bran...,Yes,"Recovered to perform SDK integration, changelo...",Same contributors (Tim Coleman / TJC continued...
1,knossos-project_knossos,The repository had several gaps of one to two ...,Yes,Recovered to apply bug fixes and maintain debu...,Same contributors (intermittent maintenance by...
2,parflow_parflow,The approximately two-month gap may have been ...,Yes,Recovered to release a new version; first post...,Same contributors (the core development team c...
3,openmd_openmd,The first commits after the gap addressed comp...,Yes,Recovered to update code compatibility for a N...,Same contributors (existing maintainers resolv...
4,project-konda_pattern-based-quality-analysis,Development appears to occur in short bursts r...,Yes,Recovered following a normal cyclical sprint g...,Same contributors (the same core developers re...
5,inab_wes-backend,Most commits after the gap were minor or cosme...,Yes,"Recovered for minor, cosmetic, and dependency ...",Same contributor (activity pre- and post-gap i...
6,wistaria_materiappsinstaller,"Branch merges surrounded the gap, suggesting t...",Yes,Recovered to update libraries and merge featur...,Same contributors (collaborative branch mergin...
7,leftygray_cascade_calculations,Commits after the gap primarily added publicat...,Yes,"Recovered to add research publications, data, ...",Same contributors (the main researcher returne...
8,wtmatlock_flanker,The final commits before the gap primarily upd...,Yes,Recovered to update test suites and perform fi...,Same contributors (the same authors returned t...
9,google_uis-rnn,"The repository is archived and read-only, and ...",Yes,Recovered briefly to apply package/dependency ...,New and existing contributors (some automated/...


In [13]:
import pandas as pd
from collections import Counter

# Ensure we have the latest commits loaded with converted dates
# Find the overall latest commit for each project
latest_commit_dates = commits.groupby('project')['commit_date'].max().reset_index()
latest_commit_dates.columns = ['project', 'latest_commit_date']

# Target cutoff date: April 1, 2025 (UTC)
cutoff_date = pd.to_datetime('2025-04-01', utc=True)

status_list = []
themes_list = []

for project in stats['project']:
    # Get the latest date for this project
    proj_latest_row = latest_commit_dates[latest_commit_dates['project'] == project]
    if proj_latest_row.empty:
        status_list.append('Inactive')
        themes_list.append('Other')
        continue

    latest_date = proj_latest_row.iloc[0]['latest_commit_date']

    # Determine Active/Inactive status
    status = 'Active' if latest_date >= cutoff_date else 'Inactive'
    status_list.append(status)

    # Fetch the last 10 commits for this project overall ordered chronologically
    proj_commits = commits[commits['project'] == project].sort_values('commit_date', ascending=False).head(10)

    # Categorize themes for these recent 10 commits
    proj_commits['Theme'] = proj_commits.apply(classify_theme, axis=1)

    # Identify the top 2 themes
    counts = Counter(proj_commits['Theme'].dropna())
    ranked = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    top_themes = ":".join(theme for theme, count in ranked[:2])
    themes_list.append(top_themes if top_themes else 'Other')

# Load the statistics file to ensure we don't overwrite previous steps
stats_df = pd.read_csv('sydlwils_project_stats.csv')

stats_df['Currentstatus'] = status_list
stats_df['RecentThemes'] = themes_list

# Save the final file
stats_df.to_csv('sydlwils_project_stats.csv', index=False)
print("Saved updated stats with Currentstatus and RecentThemes columns to sydlwils_project_stats.csv")
display(stats_df[['project', 'Currentstatus', 'RecentThemes']])

Saved updated stats with Currentstatus and RecentThemes columns to sydlwils_project_stats.csv


,project,Currentstatus,RecentThemes
0,nrel_openstudio-server,Active,Other:Bug fixes
1,knossos-project_knossos,Active,Other:Bug fixes
2,parflow_parflow,Active,Release:Branch Handling
3,openmd_openmd,Active,Bug fixes:Other
4,project-konda_pattern-based-quality-analysis,Active,Other:Branch Handling
5,inab_wes-backend,Active,Automated bot contributions:Release
6,wistaria_materiappsinstaller,Active,Branch Handling:Bug fixes
7,leftygray_cascade_calculations,Inactive,Release:Bug fixes
8,wtmatlock_flanker,Inactive,Documentation updates:Release
9,google_uis-rnn,Inactive,Dependency updates:Other


Reflections:

parflow_parflow: the first commit after the gap was the release of a new version, so I imagine the 2 month gap was preperation and planning the new version release.

inab_wes-backend: All commits after the break were minor/cosmetic fixes. Almost all commits seen were by one person, so maybe another project or a break from this project caused the break.

knossos-project_knossos: This project has several gaps spanning a month or two with not much of a pattern. All commits in this portion of the project seem to be debugging fixes that are spread randomly.

project-konda_pattern-based-quality-analysis: This project seems to commit in sprints, several in a short period. The break here was about a month, not substantial.

wistaria_materiappsinstaller: The break was surrounded by branch merging whihc means this may just be evidence of work split. First few commits after the break looked like library updating.

wtmatlock_flanker: Before the break, the last 5-6 commits were all README updates. All commits after the break were updates to testing files which seem to be one of the last things before the project looks to be done.

openmd_openmd: First commits after the break were updates to make a numpy upgrade work.

google_uis-rnn: This repo is archived and read-only. Gap was almost two years long, and all changes after break were package/requirements updates presumably to fix errors in older code as packages upgarde.

leftygray_cascade_calculations: Commits after the gap mainly included adding new publications and data as well as updating the README.

nrel_openstudio-server: Commits before the gap are all branch merges, and the commits after the break were mainly package and requirements upgrades.